# Notebook 05 — Trading Rule and Backtest

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

**Conditional on Notebook 02 DA gate passing.**

---

### Trading rule (pre-registered)

```
Signal:    conditional_p25(GB_DA_baseload | outage_state) > futures_strip
           + bid-offer spread allowance
Position:  Long GB DA Baseload block, 1 MW standard lot
           (5 MW Peak block for secondary peak test)
Sizing:    Kelly fraction on estimated edge / variance
Exit:      DA auction close
Entry:     fetch_timestamp + 15-minute execution delay
```

### Pre-registered cost scenarios

| Scenario | Round-trip cost | Notes |
|---|---|---|
| Optimistic | £0.10/MWh | Tight bid-offer, non-outage conditions |
| Base | £0.30/MWh | Typical including exchange fees |
| Conservative | £0.60/MWh | Outage day: spreads widen precisely when you want to trade |

**The strategy must clear Scenario 3 to be considered tradeable.**

Sample: 2020–2026 (post IFA2 coupling). Pre-2020 excluded due to structural break.


> 📋 **NOT RUN** — Pre-registered but not executed.
> The DA hard gate (Notebook 02) failed before this gate was reached.
> Published unmodified to demonstrate pre-registration discipline.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src import log, load, DATA_DIR, MAIN_START, MAIN_END

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


---
## 1. Load Signal and Price Data

In [ ]:
try:
    signal = load("signal_quantile")
    signal.index = pd.to_datetime(signal.index, utc=True)
    print(f"Signal loaded: {len(signal)} rows")
except FileNotFoundError:
    raise FileNotFoundError(
        "signal_quantile.parquet not found — run Notebook 04 first."
    )

gb_da = load("elexon_da_prices")[["gb_da_price"]]
gb_da.index = pd.to_datetime(gb_da.index, utc=True)

print(signal.describe())


---
## 2. Build Backtest

In [ ]:
COST_SCENARIOS = {
    "optimistic":   0.10,   # £/MWh round trip
    "base":         0.30,
    "conservative": 0.60,   # outage days — spreads widen
}

EXCHANGE_FEE = 0.03     # EPEX transaction cost £/MWh
POSITION_MW  = 1.0      # 1 MW standard lot
MAX_POSITION = 5.0      # pre-registered position ceiling

def run_backtest(signal_df, cost_rtt, position_mw=1.0, label=""):
    """
    Backtest the conditional P25 trading rule.
    
    Entry condition: conditional_p25 > current_da_price + cost_rtt
    Position: long 1 MW GB DA Baseload block
    Exit: DA auction close (PnL = actual_da_price - entry_cost)
    
    Since we're buying a forward contract at the DA clearing price,
    PnL per trade = (gb_da_actual - conditional_p25 + edge) × position_mw
    
    More precisely: we enter when conditional_p25 > strip (proxy = previous day's
    DA price as a pre-auction estimate). PnL = actual_da_price - entry_price.
    Entry price = conditional_p25 (our model's estimate of fair value).
    """
    df = signal_df.copy()
    
    # Signal fires when estimated edge exceeds cost
    df["net_edge"]    = df["estimated_edge"] - cost_rtt - EXCHANGE_FEE
    df["signal_fire"] = (df["net_edge"] > 0) & (df["unplanned_outage_mw"] > 0)
    
    trades = df[df["signal_fire"]].copy()
    
    if len(trades) == 0:
        return pd.DataFrame(), {"n_trades": 0, "label": label}
    
    # PnL per trade: actual DA price - entry price (conditional_p25 used as entry)
    # Positive PnL if actual > our estimated P25 (we bought undervalued DA)
    trades["entry_price"]  = trades["conditional_p25"]
    trades["exit_price"]   = trades["gb_da_price"]      # actual DA settlement
    trades["gross_pnl"]    = (trades["exit_price"] - trades["entry_price"]) * position_mw
    trades["cost"]         = cost_rtt * position_mw
    trades["net_pnl"]      = trades["gross_pnl"] - trades["cost"]
    trades["backtest_label"] = label
    
    return trades, build_stats(trades, label)

def build_stats(trades, label):
    if len(trades) == 0:
        return {"label": label, "n_trades": 0}
    
    pnl = trades["net_pnl"]
    daily_returns = pnl / POSITION_MW   # £/MWh — normalised
    
    sharpe = (daily_returns.mean() / daily_returns.std() * np.sqrt(252)
              if daily_returns.std() > 0 else float("nan"))
    
    cumulative = pnl.cumsum()
    drawdown   = cumulative - cumulative.cummax()
    
    return {
        "label":       label,
        "n_trades":    len(trades),
        "hit_rate":    (pnl > 0).mean(),
        "mean_pnl":    pnl.mean(),
        "total_pnl":   pnl.sum(),
        "sharpe":      sharpe,
        "max_dd":      drawdown.min(),
        "mean_edge_net": trades["net_edge"].mean(),
    }

print("Running backtests across cost scenarios...")
print()

all_stats = []
all_trades = {}

for scenario, cost in COST_SCENARIOS.items():
    trades, stats = run_backtest(signal, cost_rtt=cost, label=scenario)
    all_stats.append(stats)
    all_trades[scenario] = trades
    
    print(f"Scenario: {scenario.upper()} (£{cost}/MWh round trip)")
    if stats["n_trades"] == 0:
        print("  No trades fired — edge does not clear this cost threshold.")
    else:
        print(f"  N trades:    {stats['n_trades']}")
        print(f"  Hit rate:    {stats['hit_rate']:.1%}")
        print(f"  Mean PnL:    £{stats['mean_pnl']:.2f} per trade")
        print(f"  Total PnL:   £{stats['total_pnl']:.2f}")
        print(f"  Sharpe:      {stats['sharpe']:.3f}")
        print(f"  Max DD:      £{stats['max_dd']:.2f}")
    print()


---
## 3. Gate Decision

In [ ]:
conservative_stats = next(s for s in all_stats if s["label"] == "conservative")
base_stats         = next(s for s in all_stats if s["label"] == "base")

print("=" * 60)
print("TRADING RULE ASSESSMENT")
print("=" * 60)
print()

# The strategy must clear conservative scenario to be tradeable
c = conservative_stats
if c["n_trades"] == 0:
    tradeable = False
    print("❌  No trades fire under conservative cost assumptions.")
elif c["sharpe"] is not None and c["sharpe"] > 0 and c["total_pnl"] > 0:
    tradeable = True
    print(f"✅  Strategy clears conservative scenario (£0.60/MWh round trip).")
    print(f"   Sharpe: {c['sharpe']:.3f} | Total PnL: £{c['total_pnl']:.2f}")
else:
    tradeable = False
    print(f"❌  Strategy does not clear conservative scenario.")
    print(f"   Sharpe: {c.get('sharpe','N/A')} | Total PnL: £{c.get('total_pnl','N/A')}")
    print()
    b = base_stats
    if b["n_trades"] > 0 and b.get("sharpe",0) > 0:
        print(f"   Note: strategy passes base scenario (£0.30/MWh).")
        print(f"   Reporting as 'conditional on normal market conditions'.")
        print(f"   Outage days are NOT normal market conditions.")
        print(f"   This edge may not survive live execution.")

print()
print("Summary across scenarios:")
print(f"{'Scenario':<15} {'N trades':>10} {'Sharpe':>10} {'Total PnL':>12} {'Tradeable':>12}")
print("─" * 62)
for s in all_stats:
    tradeable_flag = (
        "✅" if s.get("n_trades",0)>0 and s.get("sharpe",None) is not None and s.get("sharpe",0)>0 and s.get("total_pnl",0)>0
        else "❌"
    )
    sharpe_str = f"{s['sharpe']:.3f}" if isinstance(s.get("sharpe"), float) and not np.isnan(s.get("sharpe",float("nan"))) else "—"
    pnl_str    = f"£{s['total_pnl']:.2f}" if s.get("n_trades",0)>0 else "—"
    print(f"{s['label']:<15} {s.get('n_trades',0):>10} {sharpe_str:>10} {pnl_str:>12} {tradeable_flag:>12}")


---
## 4. Equity Curve and Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {"optimistic": "steelblue", "base": "darkorange", "conservative": "tomato"}

# ── Equity curves ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
for scenario, trades in all_trades.items():
    if len(trades) > 0:
        cumulative = trades["net_pnl"].cumsum()
        ax.plot(cumulative.index, cumulative.values, lw=1.3,
                color=colors[scenario], label=f"{scenario} (£{COST_SCENARIOS[scenario]}/MWh)")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("Cumulative PnL — All Cost Scenarios (1 MW)")
ax.set_ylabel("£")
ax.legend(fontsize=9)

# ── Trade PnL distribution (base scenario) ───────────────────────────────────
ax = axes[0, 1]
base_trades = all_trades.get("base", pd.DataFrame())
if len(base_trades) > 0:
    ax.hist(base_trades["net_pnl"], bins=30, color="darkorange", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.axvline(base_trades["net_pnl"].mean(), color="red", lw=1.5, ls="--", label=f"Mean: £{base_trades['net_pnl'].mean():.2f}")
    ax.set_title("Trade PnL Distribution (Base Scenario)")
    ax.set_xlabel("£ per trade")
    ax.legend()

# ── Drawdown (base scenario) ──────────────────────────────────────────────────
ax = axes[1, 0]
if len(base_trades) > 0:
    cum = base_trades["net_pnl"].cumsum()
    dd  = cum - cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, color="tomato", alpha=0.5)
    ax.set_title("Drawdown (Base Scenario)")
    ax.set_ylabel("£")

# ── PnL vs outage size ────────────────────────────────────────────────────────
ax = axes[1, 1]
if len(base_trades) > 0:
    ax.scatter(base_trades["unplanned_outage_mw"]/1000,
               base_trades["net_pnl"],
               alpha=0.5, s=25, color="darkorange")
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("Outage MW (GW)")
    ax.set_ylabel("Net PnL per trade (£)")
    ax.set_title("Trade PnL vs Outage Size (Base Scenario)")

plt.suptitle("Notebook 05 — Trading Rule Backtest", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / "plot_05_backtest.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 5. Falsification — Out-of-Sample Sharpe

In [ ]:
# Pre-registered falsification 6: out-of-sample Sharpe < 0 → in-sample overfit
# Use 2025 as out-of-sample (held out entirely from model fitting)

oos_mask = signal.index >= pd.Timestamp("2025-01-01", tz="UTC")
if oos_mask.sum() > 10:
    oos_signal = signal[oos_mask]
    for scenario, cost in COST_SCENARIOS.items():
        oos_trades, oos_stats = run_backtest(oos_signal, cost_rtt=cost, label=f"OOS {scenario}")
        if oos_stats["n_trades"] > 0:
            sharpe_str = f"{oos_stats['sharpe']:.3f}" if isinstance(oos_stats.get("sharpe"), float) else "—"
            print(f"OOS {scenario}: n={oos_stats['n_trades']}, Sharpe={sharpe_str}")
            if oos_stats.get("sharpe", 0) is not None and not np.isnan(oos_stats.get("sharpe", float("nan"))) and oos_stats.get("sharpe", 0) < 0:
                print(f"  ⚠️  Negative OOS Sharpe — possible in-sample overfit (falsification 6 triggered)")
        else:
            print(f"OOS {scenario}: no trades fired")
else:
    print("Insufficient 2025 data for OOS check — re-run when more data available.")


---
## Summary

| Scenario | N trades | Sharpe | Total PnL | Tradeable |
|---|---|---|---|---|
| Optimistic (£0.10) | | | | |
| Base (£0.30) | | | | |
| Conservative (£0.60) | | | | |

*(See cell output for actual values)*

**Decision rule:** tradeable if conservative scenario clears (Sharpe > 0, PnL > 0).

Proceed to **Notebook 06** (signal decay test).
